# Build Your Medical Consultation Agent with Qwen, vLLM, and AMD MI300X GPU

Welcome to this hands-on workshop! Throughout this tutorial, we'll leverage AMD GPUs and Qwen to build a doctor-style conversational assistant that reacts to a vision model's triage summary. Key components:
- 🖥️ **vLLM** for GPU-optimized inference
- 🛠️ **Pydantic-AI** for agent and tool management
- 💬 **Qwen** for post-triage question answering

You'll learn how to set up your environment, deploy a Qwen model with vLLM, connect it to structured triage output, and build a conversational agent that can answer follow-up questions about symptoms, prescriptions, and next steps.

By the end of this workshop, you’ll have built a medical consultation assistant that can review a vision model's triage report and guide the user through the next conversation with a doctor-like tone.

Let’s dive in!

## Table of Contents

- [Step 1: Launching vLLM Server on AMD GPUs](#step1)
- [Step 2: Installing Dependencies](#step2)
- [Step 3: Create a simple instance of Pydantic-AI Agent](#step3)
- [Step 4: Write a triage helper tool for your agent](#step4)
- [Step 5: Make the agent consultative and Qwen-aware](#step5)
- [Step 6: Turn the agent into a follow-up doctor chat](#step6)
- [Step 7: Challenge](#step7)

<a id="step1"></a>

## Step 1: Launch a vLLM Server

In this workshop we are going to use [vLLM](https://github.com/vllm-project/vllm) as our inference serving engine. vLLM provides many benefits such as fast model execution, extensive list of supported models, easy to use, and best of all it's open-source. 

### Deploy Qwen3-30B-A3B Model with vLLM



Time to start your vLLM server and creating an end-point for your LLM. Let's open a terminal using your Jupyter server. Then run the following command in this terminal to start the vLLM server:

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
    --served-model-name Qwen3-30B-A3B \
    --api-key abc-123 \
    --port 8000 \
    --enable-auto-tool-choice \
    --tool-call-parser hermes \
    --trust-remote-code
```

Open another terminal and monitor the GPU utilization by running this command:

```bash
watch rocm-smi
```

Upon successful launch, your server should be accepting incoming traffic through an OpenAI-compatible API. Let's set some environment variables for our server so we can use throughout this tutorial:

In [ ]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

We can verify your model is available at the `BASE_URL` we just set by running the following command.

In [ ]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

Congratulations, you now just launched a powerful server that can serve any incoming request and allowing you to build amazing applications. Wasn't that easy?🎉 

<a id="step2"></a>

## Step 2: Installing Dependencies

We are going to use `Pydantic AI`. Let's install the dependencies:

In [ ]:
!pip install -q pydantic_ai openai     


<a id="step3"></a>

## Step 3: Create a simple instance of Pydantic-AI Agent

Let's start by creating a custom OpenAI Compatible endpoint for our agent. 


In [ ]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIModel("Qwen3-30B-A3B", provider=provider)

Let's start by creating an instance the `Agent` class from `pydantic_ai`. 


In [ ]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

It's time to test the agent. `pydantic_ai` provides multiple ways to run `Agent`. You can learn more about it [here](https://ai.pydantic.dev/agents/#running-agents).

In this workshop, we are running in `async` mode. We are going to define a helper function that allows us to quickly test our agent throughout this workshop.

In [ ]:
import asyncio

async def run_async(prompt: str) -> str:
    result = await agent.run(prompt)
    return result.output


Test the agent by calling this function.

In [ ]:
await run_async("What is the capital of France?")

Great! now that we have the basics of creating an agent instance, and connecting it to the model we started serving with vLLM earlier.

<a id="step4"></a>

## Step 4: Write a triage helper tool for your agent

LLMs naturally rely on their training data to respond to your prompts. Therefore, the agent we just defined fails to answer a medical question that falls outside of its training knowledge. Let's show this with an example:

In [ ]:
await run_async("What’s the date today?")

Now give the agent a custom tool that normalizes a triage summary before follow-up questions. This helps the Qwen layer stay grounded in the vision model's output.

In [ ]:
from pydantic_ai import Tool

@Tool
def summarize_triage_report(report: str) -> str:
    """Normalize a vision-model triage report before follow-up questions."""
    return report.strip()


The vector database lets the consultation flow reuse structured triage reports before answering patient questions.

In [ ]:
agent = Agent(
    model=agent_model,
    tools=[summarize_triage_report],
    system_prompt=(
        "You are a careful medical consultation assistant.\n"
        "Use the triage summary to answer patient follow-up questions.\n"
        "Be concise, ask clarifying questions when needed, and escalate urgently when red flags appear."
    ),
)

Let's test the agent.

In [ ]:
await run_async("Using this triage report: image_type: xray; triage_label: urgent; summary: possible chest issue. Ask me what symptoms I have and explain what I should watch for.")

Well done on building an agent with access to real-time data. 

<a id="step5"></a>

## Step 5: Add a local vector memory layer

Now that we have the core agent flow working, let's give it persistence. We are going to store triage summaries in a local vector database and retrieve similar notes before each follow-up question.

**Why this matters:** it lets the assistant remember prior triage reports, reuse previous context, and answer from the most relevant stored notes instead of only the latest prompt.

Let's build the retrieval layer:

In [ ]:
!pip install -q chromadb sentence-transformers

Now let's initialize the local vector database and the helper functions for storing and retrieving triage summaries.

In [ ]:
import hashlib
import json
import os
from typing import Any

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

VECTOR_DB_PATH = os.path.join(os.getcwd(), "medical_memory_chroma")
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

embedding_function = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)
chroma_client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
medical_collection = chroma_client.get_or_create_collection(
    name="medical_triage_notes",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"},
)


def triage_report_to_text(triage_report: dict[str, Any]) -> str:
    return "\n".join(
        [
            f"image_type: {triage_report.get('image_type', 'unknown')}",
            f"triage_label: {triage_report.get('triage_label', 'unknown')}",
            f"summary: {triage_report.get('summary', '')}",
            f"findings: {triage_report.get('findings', '')}",
            f"prescription_text: {triage_report.get('prescription_text', 'none')}",
            f"follow_up_questions: {', '.join(triage_report.get('follow_up_questions', []))}",
        ]
    )


def store_triage_report(triage_report: dict[str, Any], conversation_id: str = "default") -> str:
    document = triage_report_to_text(triage_report)
    record_id = hashlib.sha1(f"{conversation_id}:{document}".encode("utf-8")).hexdigest()
    medical_collection.upsert(
        ids=[record_id],
        documents=[document],
        metadatas=[{
            "conversation_id": conversation_id,
            "kind": "triage_report",
        }],
    )
    return record_id


def retrieve_similar_triage_notes(user_question: str, n_results: int = 3) -> list[str]:
    results = medical_collection.query(query_texts=[user_question], n_results=n_results)
    return results.get("documents", [[]])[0]


The vision notebook should do the first-pass image interpretation. This notebook now focuses on conversation, clarifying questions, and patient-friendly guidance based on both the latest triage result and the most similar stored notes from the vector database.

In [ ]:
agent = Agent(
    model=agent_model,
    tools=[summarize_triage_report],
    system_prompt=(
        "You are a careful medical consultation assistant.\n"
        "Use the triage summary to answer patient follow-up questions.\n"
        "Be concise, ask clarifying questions when needed, and escalate urgently when red flags appear."
    ),
)


Great, let's see if the agent can respond with follow-up questions and escalation guidance using both the current report and retrieved memory.

In [ ]:
await run_async("Using this triage report: image_type: xray; triage_label: urgent; summary: possible chest issue. Ask me what symptoms I have and explain what I should watch for.")

The consultation flow is now ready to take a structured medical summary from the vision notebook and turn it into a useful doctor-style conversation.

<a id="step6"></a>

## Step 6: Turn the agent into a follow-up doctor chat

The vision notebook should do the first-pass image interpretation. This notebook now focuses on conversation, clarifying questions, and patient-friendly guidance based on that structured result.

We can keep the flow simple: pass the triage text directly into the Qwen agent and ask it to explain the next step.

In [ ]:
print("No extra system packages are needed for the follow-up consultation demo.")

The consultation flow does not require Node.js for its core path. The next section just shows a simple triage-to-Qwen handoff.

In [ ]:
def build_consultation_prompt(triage_report: dict[str, Any], user_question: str) -> str:
    current_report_text = triage_report_to_text(triage_report)
    relevant_notes = retrieve_similar_triage_notes(user_question)
    notes_block = "\n\n".join(f"- {note}" for note in relevant_notes) if relevant_notes else "No similar stored notes found."

    return (
        "You are a careful medical consultation assistant.\n"
        "Use the current triage report and the retrieved notes from the local vector database.\n\n"
        f"Current triage report:\n{current_report_text}\n\n"
        f"Retrieved notes:\n{notes_block}\n\n"
        f"User question: {user_question}\n\n"
        "Respond in a calm doctor-style tone, ask only the most relevant follow-up questions, and mention red flags if the report suggests urgency."
    )


In this part of the workshop we are going to build a medical consultation assistant that can explain a triage summary, answer follow-up questions, and help the user decide whether they need urgent care. We can now build on top of what we have so far and add a structured follow-up flow to our agent.

In [ ]:
triage_report = {
    "image_type": "xray",
    "triage_label": "urgent",
    "summary": "Possible chest issue",
    "findings": "chest opacity; needs review",
    "prescription_text": "none",
    "follow_up_questions": ["Do you have chest pain?", "Are you short of breath?", "Do you have fever or cough?"],
}


Let's update our agent for doctor-style consultation.

In [ ]:
follow_up_question = "What symptoms should I ask about next, and when should I escalate urgently?"
record_id = store_triage_report(triage_report)
consultation_prompt = build_consultation_prompt(triage_report, follow_up_question)
print(f"Stored triage report: {record_id}")
print(consultation_prompt)


Finally, let's try the assistant with a triage summary and see if it can guide a follow-up conversation using the stored vector memory.

In [ ]:
await run_async(consultation_prompt)

## Step 7: Challenge - Expand the consultation flow

**Task:** Add a safety-check or appointment-booking MCP server from any available source you can find.

1. Launch the additional server
2. Add it to the agent's tools
3. Make the agent suggest the best next step based on the triage summary, the user's symptoms, and any retrieved memory

Happy coding! If you encounter issues or have questions, don’t hesitate to ask or raise an issue on our [Github page](https://github.com/ROCm/gpuaidev)!